<a href="https://colab.research.google.com/github/Gakwaya011/athlete-recovery-AI/blob/main/ML/notebooks/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Environment Initialization
# Silently install system tools (zstd) and the RAG technology stack.
!apt-get update -qq > /dev/null
!apt-get install -y -qq zstd > /dev/null
!pip install -q langchain langchain-community langchain-core chromadb sentence-transformers pypdf langchain-text-splitters --no-cache-dir

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 116.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 288.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 399.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 416.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 297.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 207.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 166.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 222.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 214.2 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [3]:
import os
from google.colab import drive

# 2. Storage & Directory Setup
# Mount Google Drive to ensure persistent storage of the Vector Database.
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Define and create the project architecture
BASE_DIR = '/content/drive/MyDrive/Capstone_RAG'
PDF_DIR = os.path.join(BASE_DIR, 'sports_pdfs')
DB_DIR = os.path.join(BASE_DIR, 'science_db')

os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(DB_DIR, exist_ok=True)

print(f"✅ Project Environment Active: {BASE_DIR}")

✅ Project Environment Active: /content/drive/MyDrive/Capstone_RAG


In [4]:
import os
import warnings
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Suppress warnings for a clean presentation output
warnings.filterwarnings("ignore")

def get_vector_store(pdf_path, db_path):
    print("🧠 Initializing Knowledge Base...")
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    # Smart Check: Load existing DB if available to save processing time
    if os.path.exists(db_path) and os.listdir(db_path):
        print(f"✅ Loaded existing Vector Database from: {db_path}")
        return Chroma(persist_directory=db_path, embedding_function=embeddings)

    # Fallback: Build new DB only if missing
    print("⚙️ Indexing PDFs (First Run Only)...")
    loader = PyPDFDirectoryLoader(pdf_path)
    documents = loader.load()

    if not documents:
        print("⚠️ Warning: No PDFs found. Please upload to 'sports_pdfs'.")
        return None

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = text_splitter.split_documents(documents)

    vector_db = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_path)
    print(f"✅ Database built successfully with {len(chunks)} knowledge chunks.")
    return vector_db

# Initialize the vector store
vector_store = get_vector_store(PDF_DIR, DB_DIR)

🧠 Initializing Knowledge Base...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Loaded existing Vector Database from: /content/drive/MyDrive/Capstone_RAG/science_db


In [5]:
import subprocess
import time
import os

def setup_inference_engine():
    # 1. Smart Install: Only install if missing
    if os.path.exists("/usr/local/bin/ollama"):
        print("✅ Ollama engine is already installed.")
    else:
        print("⚙️ Installing Ollama engine...")
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # 2. Smart Server Start: Only start if not running
    # We try to ping the local server port to see if it's alive
    try:
        subprocess.check_output(["curl", "-s", "localhost:11434"])
        print("✅ Inference server is already running in background.")
    except:
        print("🚀 Starting background server...")
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(5) # Allow server to bind to port

    # 3. Smart Model Pull: Only pull if missing
    # We list the models first to save bandwidth
    try:
        installed_models = subprocess.check_output(["ollama", "list"]).decode("utf-8")
        if "llama3:latest" in installed_models:
            print("✅ Llama 3 model is already loaded and ready.")
            return
    except:
        pass

    print("⏳ Downloading Llama 3 weights (This may take a moment)...")
    subprocess.run(["ollama", "pull", "llama3"], check=True)
    print("✅ Model initialized and ready.")

setup_inference_engine()

⚙️ Installing Ollama engine...
🚀 Starting background server...
⏳ Downloading Llama 3 weights (This may take a moment)...
✅ Model initialized and ready.


In [10]:
import pandas as pd
import os
import re
from langchain_community.llms import Ollama
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. SETUP PATHS
BASE_DIR = '/content/drive/MyDrive/Capstone_RAG'
DB_DIR = os.path.join(BASE_DIR, 'science_db')
CSV_PATH = os.path.join(BASE_DIR, 'East_Africa_Food_Dataset.csv')

# 2. LOAD RESOURCES (Heavy lifting - run once)
print("🔄 Loading Science Brain, Food Data, and AI...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)
food_df = pd.read_csv(CSV_PATH)
llm = Ollama(model="llama3")

print("✅ All Systems Ready.")

🔄 Loading Science Brain, Food Data, and AI...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ All Systems Ready.


In [12]:
class FinalCoach:
    def __init__(self, vector_store, food_df, llm):
        # This grabs the resources loaded in Cell 1
        self.retriever = vector_store.as_retriever(search_kwargs={"k": 3})
        self.llm = llm
        self.food_df = food_df
        self._parse_dataset()

    def _parse_dataset(self):
        def extract_nutrient(text, type_name):
            match = re.search(rf'([0-9.]+)\s*g\s*{type_name}', str(text), re.IGNORECASE)
            return float(match.group(1)) if match else 0.0
        self.food_df['Carbs_Parsed'] = self.food_df['Macros per 100g (C / P / F)'].apply(lambda x: extract_nutrient(x, 'Carbs'))
        self.food_df['Protein_Parsed'] = self.food_df['Macros per 100g (C / P / F)'].apply(lambda x: extract_nutrient(x, 'Pro'))

    def _get_precalculated_menu(self, target_carbs):
        solids = self.food_df[
            self.food_df['Primary Category'].str.contains('Starch|Root|Tuber|Cereal|Rice', case=False, na=False) &
            ~self.food_df['Food Item'].str.contains('Porridge|Flour|Uji|Powder', case=False, na=False)
        ]
        solids = solids.sort_values(by='Carbs_Parsed', ascending=False).head(5)

        proteins = self.food_df[
            self.food_df['Primary Category'].str.contains('Protein|Legume|Bean|Pea|Fish', case=False, na=False) &
            ~self.food_df['Food Item'].str.contains('Powder|Shake', case=False, na=False)
        ]
        proteins = proteins.sort_values(by='Protein_Parsed', ascending=False).head(5)

        liquids = self.food_df[self.food_df['Primary Category'].str.contains('Beverage|Fruit', case=False, na=False)]
        liquids = liquids.sort_values(by='Carbs_Parsed', ascending=False).head(4)

        menu = "--- CARB OPTIONS (MATH IS ALREADY DONE, COPY THESE) ---\n"
        for _, row in solids.iterrows():
            if row['Carbs_Parsed'] > 0:
                raw_grams = (target_carbs / row['Carbs_Parsed']) * 100
                rounded_grams = int(round(raw_grams, -1))
                clean_name = row['Food Item'].replace('Raw ', '').replace('Dry ', '')
                menu += f"- {clean_name}: Eat {rounded_grams}g\n"

        menu += "\n--- PROTEIN OPTIONS ---\n"
        for _, row in proteins.iterrows():
            clean_name = row['Food Item'].replace('Raw ', '').replace('Dry ', '')
            menu += f"- {clean_name}\n"

        menu += "\n--- HYDRATION ---\n"
        for _, row in liquids.iterrows():
            menu += f"- {row['Food Item']}\n"

        return menu

    def _validate_inputs(self, query):
        weight_match = re.search(r'(\d+)\s*(kg|lbs)', query, re.IGNORECASE)
        if not weight_match:
            print("\n⚠️ WAIT! I didn't see your weight.")
            weight = input("   >> Please enter your weight (e.g., 65kg): ")
            return f"{query}. My weight is {weight}.", float(re.search(r'\d+', weight).group())
        return query, float(weight_match.group(1))

    def run(self):
        print("\n💬 TELL ME ABOUT YOUR WORKOUT:")
        initial_query = input("   >> ")

        full_query, weight = self._validate_inputs(initial_query)
        target_carbs = round(weight * 1.2, 1)

        print(f"\n⚙️ PYTHON LOGIC: Detected {weight}kg. Target = {target_carbs}g Carbs.")
        print("🧠 Coach is preparing your meal plan...")

        docs = self.retriever.invoke(full_query)
        science_rules = "\n".join(doc.page_content for doc in docs)
        menu_context = self._get_precalculated_menu(target_carbs)

        template = f"""
        You are "Coach K", an elite Sports Nutritionist.

        USER CONTEXT: "{full_query}"
        TARGET CARBS: {target_carbs}g

        PRE-CALCULATED LOCAL MENU (Math is already done):
        {menu_context}

        TASK:
        Create a friendly, highly professional recovery meal plan.

        **CRITICAL RULES:** 1. YOU MUST COPY THE EXACT "Eat Xg" PORTION SHOWN IN THE MENU. DO NOT do any math. DO NOT divide the target.
        2. You MUST include Hydration. Always recommend Water.

        OUTPUT FORMAT (Use this exact structure):

        Niceee 🔥 that’s a serious workout.

        **✅ What Your Body Needs Right Now**
        * **Carbs:** {target_carbs}g (to refill energy fast, based on UEFA guidelines of 1.2g/kg)
        * **Protein:** to repair muscle damage
        * **Fluids:** to recover properly

        Let’s keep it simple, local, and effective.

        **🥗 Your Post-Workout Recovery Plate**
        * **🍚 Main Carb:** [Copy the EXACT line from the Carb Menu. e.g. "Fried Plantain: Eat 260g"]
        * **🍗 Protein:** [Pick a Protein from Menu] (e.g., Stewed or Grilled)
        * **💧 Hydration:** 500ml of Water, plus [Pick a Drink from Menu if available]

        **💡 Coach's Tip**
        Time is of the essence! Consume your recovery meal and drink plenty of water within 30-60 minutes after your workout.
        """

        response = self.llm.invoke(template)
        print("\n" + "="*60)
        print(response)
        print("="*60)

# --- EXECUTE ---
agent = FinalCoach(vector_store, food_df, llm)
agent.run()


💬 TELL ME ABOUT YOUR WORKOUT:
   >> I played  basketball for 2hours and did some push ups 

⚠️ WAIT! I didn't see your weight.
   >> Please enter your weight (e.g., 65kg): 68kg

⚙️ PYTHON LOGIC: Detected 68.0kg. Target = 81.6g Carbs.
🧠 Coach is preparing your meal plan...

Niceee 🔥 that’s a serious workout.

**✅ What Your Body Needs Right Now**
* **Carbs:** 81.6g (to refill energy fast, based on UEFA guidelines of 1.2g/kg)
* **Protein:** to repair muscle damage
* **Fluids:** to recover properly

Let’s keep it simple, local, and effective.

**🥗 Your Post-Workout Recovery Plate**
* **🍚 Main Carb:** Sweet Potato: Eat 328g
* **🍗 Protein:** Boiled Chicken Breast
* **💧 Hydration:** 500ml of Water, plus Water
